# Step 2: A2A PEMDAS
In Step 1, we used A2A discovery + specialist selection (A2A Olympics).

Now, we demonstrate agent coordination through multiple agents focused on arithmetic.

What you get
- 2 deterministic A2A-ish agent servers (Add, Multiply)
- 1 intelligent agent (Evaluate) which breaks down an arithmetic problem and assigns tasks to the deterministic agents

Notes
- This tool implements a minimal subset of the A2A patterns: Agent Card discovery + JSON-RPC `message/send` + `tasks/get`.
- It follows the published examples for the Agent Card path `/.well-known/agent.json` and the JSON-RPC `message/send` shape.


## 1) Write the local agent servers
Each agent exposes:
- `GET /.well-known/agent.json` (Agent Card)
- `POST /a2a` JSON-RPC with:
  - `message/send`
  - `tasks/get`

Ports:
- Evaluator: 8100
- Add: 8101
- Mult: 8102


In [1]:
from dotenv import load_dotenv
load_dotenv()

import os
import sys
from pathlib import Path

def find_repo_root(start: Path) -> Path:
    for p in [start] + list(start.parents):
        if (p / "pyproject.toml").exists():
            return p
    raise RuntimeError("Could not find repo root (pyproject.toml not found).")

REPO_ROOT = find_repo_root(Path.cwd())
SRC_DIR = REPO_ROOT / "src"
APP_DIR = SRC_DIR / "demos" / "a2a_pemdas"
sys.path.insert(0, str(SRC_DIR.resolve()))

from a2a.core import (
    start_uvicorn, wait_http_ok, tail, stop_proc,
     AgentRegistry, fetch_agent_card, A2AHttpAgent,
)

# ---- CONFIG ----
HOST = "127.0.0.1"

ADD_PORT = 8101
MULT_PORT = 8102
EVAL_PORT = 8100

agents = {
    "evaluator": ("evaluate_agent:app", EVAL_PORT),
    "add":       ("add_agent:app", ADD_PORT),
    "mult":      ("mult_agent:app", MULT_PORT),
}

# Pass env down to child processes
child_env = dict(os.environ)

# Ensure EVAL points at the running agent URLs
child_env["ADD_BASE_URL"] = f"http://{HOST}:{ADD_PORT}"
child_env["MULT_BASE_URL"] = f"http://{HOST}:{MULT_PORT}"

procs = {}

for name, (spec, port) in agents.items():
    url = f"http://{HOST}:{port}/.well-known/agent.json"
    try:
        wait_http_ok(url, timeout_s=1)
        print(f"✅ {name} already running on {port}")
        continue
    except Exception:
        pass

    print(f"▶ starting {name} on {port} ...")
    p = start_uvicorn(spec, HOST, port, cwd=APP_DIR, env=child_env)
    procs[name] = p

    try:
        wait_http_ok(url, timeout_s=20)
        print(f"✅ {name} up: {url}")
    except Exception as e:
        print(f"❌ {name} failed to start: {e}")
        print(tail(p, n=80))
        stop_proc(p)
        raise

print("Done. Running:", ", ".join(list(procs.keys())))


▶ starting evaluator on 8100 ...
✅ evaluator up: http://127.0.0.1:8100/.well-known/agent.json
▶ starting add on 8101 ...
✅ add up: http://127.0.0.1:8101/.well-known/agent.json
▶ starting mult on 8102 ...
✅ mult up: http://127.0.0.1:8102/.well-known/agent.json
Done. Running: evaluator, add, mult


In [2]:
agent_urls = {
    "evaluator": "http://127.0.0.1:8100",
    "add":       "http://127.0.0.1:8101",
    "mult":     "http://127.0.0.1:8102",
}
cards = [fetch_agent_card(u) for u in agent_urls.values()]
reg = AgentRegistry(cards)

In [3]:
import json
from a2a.client import a2a_send
from a2a.envelope import make_request, make_endpoint

SRC = make_endpoint(name="TEST", version="0.0.0")
DEST = make_endpoint(name="EVALUATE", url="http://127.0.0.1:8100", skill="pemdas.eval")

req = make_request(
    message_type="pemdas.eval:v1",
    payload={"type":"pemdas.eval:v1", "expression":"(2+3)*(7-4)/2", "max_steps": 50},
    source=SRC,
    dest=DEST,
)

task = a2a_send("http://127.0.0.1:8100", req_id=req["request_id"], msg_obj=req)

meta = task["meta"]                      # a2a.response:v1
result_payload = meta["payload"]          # pemdas.eval.result:v1

print("final:", result_payload["final"])
print("result:", result_payload["result"])
print("steps:", result_payload["step_count"])
print(json.dumps(result_payload["steps"], indent=2))


final: 15/2
result: {'type': 'rat:v1', 'n': 15, 'd': 2}
steps: 4
[
  {
    "i": 1,
    "before": "(2+3)",
    "call": {
      "agent": "ADD",
      "dest_url": "http://127.0.0.1:8101",
      "trace_id": "c1591824-9583-402e-9097-b6db2cb5e3eb",
      "message_type": "pemdas.add:v1",
      "a": {
        "type": "rat:v1",
        "n": 2,
        "d": 1
      },
      "b": {
        "type": "rat:v1",
        "n": 3,
        "d": 1
      },
      "request_id": "c0f68967-9b4d-4db3-b244-3e0589eec853",
      "latency_ms": 19.426
    },
    "result": {
      "type": "rat:v1",
      "n": 5,
      "d": 1
    },
    "after": "((5*(7-4))/2)"
  },
  {
    "i": 2,
    "before": "(7-4)",
    "call": {
      "agent": "ADD",
      "dest_url": "http://127.0.0.1:8101",
      "trace_id": "c1591824-9583-402e-9097-b6db2cb5e3eb",
      "message_type": "pemdas.add:v1",
      "a": {
        "type": "rat:v1",
        "n": 7,
        "d": 1
      },
      "b": {
        "type": "rat:v1",
        "n": -4,
        

In [4]:
# Stop processes when done
for name, p in procs.items():
    print(f"■ stopping {name} ...")
    stop_proc(p)

■ stopping evaluator ...
■ stopping add ...
■ stopping mult ...
